# Experiments 33-40 -  ***OPTUNA SEARCH***
Pruebas de validación ajustando Confidence y IOU setup para Non-Maximum Suppression (NMS).

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Weights:** Exp. 26 *(Full fine-tuned, no freeze)*
- **Experiments:**
    1. Optuna hyperparam search: `conf=0.15/0.30` | `iou=0.3/0.6`
- **Reference:** Default parameters: `conf=0.25` | `iou=0.6`

## Init

In [1]:
import os
import shutil
import fnmatch
import pickle

In [25]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 24.4 MB/s eta 0:00:00


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.5/983.5 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

## Helper Functions

In [3]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [4]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [5]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [6]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [7]:
# Load/Download prediction results (BB + confidence) as .pkl file

def save_results(results, filename):
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

def load_results(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [8]:
def save_on_cloud(source: str, destination: str):
    """
    Saves a folder to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder.
        destination (str): The path to the destination folder (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source Folder
        if not os.path.exists(destination):
            os.makedirs(destination)

        # 2. Copy the Folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("✅ Folder copied successfully:\n  ",source,"\n  -->",destination)

    except Exception as e:
        print(f"❌ An error occurred: {e}")

### Graph functions

In [9]:
cm = lambda results: results.confusion_matrix.matrix.tolist() if hasattr(results, 'confusion_matrix') and hasattr(results.confusion_matrix, 'matrix') else None

In [10]:
import json

def save_json(results):
  try:
    data_to_store = {
        "confusion_matrix": cm(results),
        "results_dict": results.results_dict,
        "speed": results.speed
    }

    # Convert the Python dictionary to a JSON string
    json_data = json.dumps(data_to_store, indent=4)

    # You can now save this JSON string to a file
    folder = str(results.save_dir)
    with open(f"/content/{folder}/results.json", "w") as f:
        f.write(json_data)

    print("✅ JSON file stored in:", folder)

  except Exception as e:
      print(f"❌ An error occurred: {e}")

  #return json_data


In [11]:
def gimme_metrics(results):
  matrix = cm(results)
  total_det = sum(sum(value) for value in matrix)
  percentages = []
  for row in matrix:
      values_percentages = []
      for value in row:
          if total_det != 0:
              percentage = (value / total_det) * 100
          else:
              percentage = 0.0
          values_percentages.append(f"{percentage:.2f}%")
      percentages.append(values_percentages)

  print("Total objects detected:", total_det)
  print("Confusion matrix:")
  for row in percentages:
      print(row)


In [ ]:
def numoji(numero):
  """
  Convierte un número entero del 1 al 10 a su emoji correspondiente.

  Args:
    numero: Un entero entre 1 y 10.

  Returns:
    Un string con el emoji correspondiente al número, o "0️⃣" si el número
    está fuera del rango.
  """
  if 0 <= numero <= 10:
    emoji_map = {
        0: "0️⃣",
        1: "1️⃣",
        2: "2️⃣",
        3: "3️⃣",
        4: "4️⃣",
        5: "5️⃣",
        6: "6️⃣",
        7: "7️⃣",
        8: "8️⃣",
        9: "9️⃣",
        10: "🔟"
    }
    return emoji_map[numero]
  else:
    return "*️⃣"

# Datasets builder

## Importing from Drive

In [12]:
!rm -rf /content/sample_data

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

3.5m.v3i.yolov8.640px	      best_e26.pt  models
3.5m.v3i.yolov8.640px.aug.v1  Inference    runs


In [15]:
drive_path = '/content/drive/MyDrive/YOLO'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

There are 6 dataset options:


['3.5m.v3i.yolov8.640px',
 'Inference',
 'models',
 'runs',
 '3.5m.v3i.yolov8.640px.aug.v1',
 'best_e26.pt']

In [16]:
choose_dataset = 1
index = choose_dataset - 1
model = os.listdir(drive_path)[index]
print("Chosen model:", model)

Chosen model: 3.5m.v3i.yolov8.640px


***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [17]:
# Option 2 (download just the needed dataset)
cloud_path = f"/content/drive/MyDrive/YOLO/{model}/"
local_path = f"/content/YOLO/"
!mkdir $local_path
!cp -r $cloud_path $local_path
src_folder = f"/content/YOLO/{model}"

## Download model

In [18]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [19]:
# Load stored model (Exp. 26)
model = YOLO("/content/drive/MyDrive/YOLO/best_e26.pt")

# Finetuning

### Training optimization

In [20]:
# Libera memoria de la GPU en caso de OOM error
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

9

In [ ]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Info

In [21]:
!nvidia-smi

Fri Apr 25 00:23:06 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
!yolo version

8.3.115


-----
## Experiment 41
### *Full fine-tuned (no freeze) | Hyperparameters serach*
Applying optuna for finding values of `conf` & `iou` that maximize F1-score.

In [90]:
average_time = 12 # average time per optuna experiment measured earlier

10.90515339

### Validation

In [82]:
import optuna

def objective(trial):

    # Suggest values for conf and iou
    conf_threshold = trial.suggest_float("conf", 0.15, 0.3) # Define a reasonable range
    iou_threshold = trial.suggest_float("iou", 0.3, 0.6)   # Define a reasonable range
    emoji_number = "".join([numoji(int(i)) for i in str(trial.number)])
    print(f"\n\n{emoji_number} Trial {trial.number}: Trying conf={conf_threshold:.4f}, iou={iou_threshold:.4f}")

    try:
        # Run validation with the suggested hyperparameters
        # Disable save_json unless you really need the files,
        # to avoid filling up the disk during optimization.
        # verbose=False to reduce output during Optuna trials,
        # unless you need to debug each trial.
        results = model.val(
            data="/content/YOLO/3.5m.v3i.yolov8.640px/data.yaml",
            batch=64,
            conf=conf_threshold,
            iou=iou_threshold,
            verbose=True, # Keep verbose to see detailed output
            save_json=True # Save JSON
        )

        # Display confusion matrix and store results as JSON
        print()
        gimme_metrics(results)
        print()
        save_json(results)
        print()

        # Extract the F1-Score.
        if hasattr(results, 'results_dict') and results.results_dict is not None:
          precision = results.results_dict['metrics/precision(B)']
          recall = results.results_dict['metrics/recall(B)']

          if precision is not None and recall is not None and (precision + recall) > 0:
              f1_score = 2 * (precision * recall) / (precision + recall)
              print(f"Trial {trial.number}: Calculated F1@0.5 = {f1_score:.4f} (P={precision:.4f}, R={recall:.4f})")
          else:
              print(f"❌ Trial {trial.number}: Could not find/calculate F1@0.5, precision@0.5 or recall@0.5 in results.metrics.")
              f1_score = 0.0 # Assign a low score if metric is missing
        else:
          print(f"❌ Trial {trial.number}: Could not access metrics from results object directly.")
          f1_score = 0.0 # Return 0.0 if metrics cannot be accessed

        return f1_score

    except Exception as e:
        print(f"Trial {trial.number}: An error occurred during validation: {e}")
        # Retornar un valor bajo para indicar que este conjunto de hiperparámetros
        # probablemente no es bueno o causó un error.
        return 0.0

In [93]:
import logging
import sys
#import os

# Optional: Configure logging for Optuna
logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# Define the storage path for the Optuna study
# Using a local SQLite database file
db_path = "sqlite:///optuna_yolov8_f1_study.db"
study_name = "yolov8_f1_optimization"

# The study progress is automatically saved to 'optuna_yolov8_f1_study.db'
# in the same directory where you run the script.

In [96]:
# Option for setting max n_trials with GPU resources
gpu_limit = 1
n_trials = round(gpu_limit*3600/average_time)

print(f"Starting training for {gpu_limit} hour limit")
print(f"Optuna will run {n_trials}")

Starting training for 1 hour limit
Optuna will run 330


In [98]:
import time

# Define manually the number of trials to do (optional)
# n_trials = 3 # You can start with a small number, e.g., 50 or 100

# --- Study Creation ---
# Check if the study already exists. If so, load it; otherwise, create a new one.
# This allows resuming the optimization later.
try:
    # Load the existing study
    study = optuna.load_study(study_name=study_name, storage=db_path)
    print(f"Resuming existing study '{study_name}' from {db_path}")
except KeyError:
    # Create a new study if it doesn't exist
    study = optuna.create_study(study_name=study_name, storage=db_path, direction="maximize")
    print(f"Created a new study '{study_name}' at {db_path}")

# --- Study Execution ---
print(f"Running Optuna optimization for F1-score ({n_trials} trials)...")
# Measuring experiment time
start_time = time.perf_counter_ns()

# Run the optimization
# Increase n_trials for a more exhaustive search.
study.optimize(objective, n_trials=n_trials)

# Stop time measurment
end_time = time.perf_counter_ns()

# --- Study Results Analytics ---
# Display the best hyperparameters and the best F1 value found
print("\nOptimization finished.")
print("Best hyperparameters: ", study.best_params)
print("Best F1-Score: ", study.best_value)
print()

# You can access the best trial if you need more details
best_trial = study.best_trial
print(f"Best trial: Number {best_trial.number}, Value {best_trial.value}")
print("Hyperparameters of the best trial: ", best_trial.params)

elapsed_time_ns = end_time - start_time
elapsed_time_s = elapsed_time_ns / 1e9
average_time = elapsed_time_s/n_trials

print(f"\n\nElapsed time: {elapsed_time_s:.2f} seconds for {n_trials}")
print(f"Average time: {average_time:.3f} seconds")

[I 2025-04-25 01:22:57,040] A new study created in RDB with name: yolov8_f1_optimization


Created a new study 'yolov8_f1_optimization' at sqlite:///optuna_yolov8_f1_study.db
Running Optuna optimization for F1-score (330 trials)...


0️⃣ Trial 0: Trying conf=0.2923, iou=0.5883
Ultralytics 8.3.115 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2119.1±1021.5 MB/s, size: 173.8 KB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.22s/it]


                   all        108       2409      0.579      0.537      0.537      0.206
Speed: 7.7ms preprocess, 22.5ms inference, 0.0ms loss, 2.4ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val


[I 2025-04-25 01:23:07,636] Trial 0 finished with value: 0.5572074983839689 and parameters: {'conf': 0.29229386723714673, 'iou': 0.5882893578850794}. Best is trial 0 with value: 0.5572074983839689.


...

                   all        108       2409      0.576      0.543      0.538      0.204
Speed: 5.2ms preprocess, 23.9ms inference, 0.0ms loss, 2.7ms postprocess per image
Saving runs/detect/val206/predictions.json...
Results saved to runs/detect/val206


[I 2025-04-25 02:00:31,854] Trial 205 finished with value: 0.5588188936158949 and parameters: {'conf': 0.23073000743013156, 'iou': 0.5614255402344072}. Best is trial 51 with value: 0.5596909081729791.



Total objects detected: 3344.0
Confusion matrix:
['43.96%', '27.96%']
['28.08%', '0.00%']

✅ JSON file stored in: runs/detect/val206

Trial 205: Calculated F1@0.5 = 0.5588 (P=0.5756, R=0.5430)


2️⃣0️⃣6️⃣ Trial 206: Trying conf=0.2220, iou=0.5840
Ultralytics 8.3.115 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2388.9±1268.1 MB/s, size: 188.7 KB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):  50%|█████     | 1/2 [00:05<00:05,  5.90s/it]
[W 2025-04-25 02:00:38,139] Trial 206 failed with parameters: {'conf': 0.2219794400376056, 'iou': 0.5839908717188285} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "<ipython-input-82-7dd444f2f570>", line 17, in objective
    results = model.val(
              ^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/ultralytics/engine/model.py", line 627, in val
    validator(model=self.model)
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/_contextlib.py", line 116, in decorat

KeyboardInterrupt: 

## Extract results for best hyperparams

In [99]:
# Validate the model with Optuna's best parameters
best_conf = study.best_params['conf']
best_iou = study.best_params['iou']

results = model.val(data="/content/YOLO/3.5m.v3i.yolov8.640px/data.yaml",
          batch=64,
          conf=best_conf,
          iou=best_iou,
          verbose=True,
          save_json=True)

Ultralytics 8.3.115 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2155.7±1000.3 MB/s, size: 170.1 KB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.14s/it]


                   all        108       2409      0.576      0.545      0.538      0.204
Speed: 5.0ms preprocess, 24.3ms inference, 0.0ms loss, 2.5ms postprocess per image
Saving runs/detect/val208/predictions.json...
Results saved to runs/detect/val208


In [100]:
print(best_conf, best_iou)

0.22408185715037293 0.5699950001507388


### Metrics

In [101]:
gimme_metrics(results)

Total objects detected: 3365.0
Confusion matrix:
['44.07%', '28.41%']
['27.52%', '0.00%']


In [102]:
save_json(results)

✅ JSON file stored in: runs/detect/val208


## Save all results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/optuna_yolov8_f1_study.db', destination='/content/drive/MyDrive/save/')

In [103]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save/
